# Tianshou API  
This notebook contains the following classes:
- A multi-agent compatible Grid World environment, where each agent is tasked with collecting their own set of coins with a limited field of view. The number of agents, number of coins, and field of view can be customized.
- A wrapper for that environment which creates a centralized policy that dictates movements for all the agents together.
- A network that can be used to train agents within either environment.
- A custom logger that prints the average reward attained during training every epoch.

## Installations

In [2]:
!pip install tianshou==0.5.0
!pip install gymnasium
!pip install "gym==0.21.0"
!pip install pygame
!pip install torch
!pip install pettingzoo

  Using cached gym-0.21.0.tar.gz (1.5 MB)
  Preparing metadata (setup.py) ... done
Requested gym==0.21.0 from https://files.pythonhosted.org/packages/4b/48/920cea66177b865663fde5a9390a59de0ef3b642ad98106ac1d8717d7005/gym-0.21.0.tar.gz has invalid metadata: Expected end or semicolon (after version specifier)
    opencv-python>=3.
                 ~~~^
Please use pip<24.1 if you need to use this version.
ERROR: Could not find a version that satisfies the requirement gym==0.21.0 (from versions: 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.1.0, 0.1.1, 0.1.2, 0.1.3, 0.1.4, 0.1.5, 0.1.6, 0.1.7, 0.2.0, 0.2.1, 0.2.2, 0.2.3, 0.2.4, 0.2.5, 0.2.6, 0.2.7, 0.2.8, 0.2.9, 0.2.10, 0.2.11, 0.2.12, 0.3.0, 0.4.0, 0.4.1, 0.4.2, 0.4.3, 0.4.4, 0.4.5, 0.4.6, 0.4.8, 0.4.9, 0.4.10, 0.5.0, 0.5.1, 0.5.2, 0.5.3, 0.5.4, 0.5.5, 0.5.6, 0.5.7, 0.6.0, 0.7.0, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.8.0.dev0, 0.8.0, 0.8.1, 0.8.2, 0.9.0, 0.9.1, 0.9.2, 0.9.3, 0.9.4, 0.9.5, 0.9.6, 0.9.7, 0.10.0, 0.10.1, 0.10.2, 0.10.3, 0.10.4, 0.10.5

## Imports

In [3]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils import seeding
import numpy as np
import pygame
from IPython.display import clear_output, display
from PIL import Image
from pettingzoo.utils import AECEnv, agent_selector
from copy import deepcopy
from typing import Optional, Tuple
import torch

import torch.nn as nn
import torch.nn.functional as F

## Multi-Agent Grid World  
This class implements a grid world where every agent learns and acts independently. Each agent's goal is to gather an individual set of coins scattered across the grid, while dealing with a limited range of vision.

In [4]:
class MultiAgentGridWorld(AECEnv):
    """
    Class that represents the Grid World in an environment where every agent learns independently.
    """
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, size=8, num_agents=2, num_coins=2, view_radius=2, render_mode=None):
        """
        Initializes the Grid World, creates observation and action spaces, and sets seed.
        
        :param size: the height and width of the grid
        :param num_agents: the number of agents in the grid world
        :param num_coins: the number of coins per agent
        :param view_radius: how many spaces away an agent is able to "see" in each direction
        :param render_mode: used to display the grid in a Jupyter Notebook
        """
        self.size = size
        self.num_coins = num_coins      
        self._view_radius = view_radius  
        
        self.max_steps = max(2 * size * size, 100)
        self.current_step = 0
        
        self.possible_agents = [f"agent{i}" for i in range(num_agents)]
        self.agents = self.possible_agents[:] # Active agents, for environments where agents can be terminated
        
        self.agent_positions = None
        self.coin_positions = None
        
        # Observation space 
        view_area = (2 * view_radius + 1)
        self.observation_spaces = {
            agent: spaces.Dict({
                # Local observations in the form of 5 channels
                # Channels: 0 = walls (out of bounds or agents), 1 = whether a space is already visited, 2 = coins
                # 3 = agent's X coordinate, 4 = agent's Y coordinate
                "local":spaces.Box(low=0, high=1, shape=(5, view_area, view_area), dtype=np.float32), 
                # Non-local knowledge: global location, individual number of coins left, total number of coins left
                "knowledge":spaces.Box(low=0, high=self.size, shape=(4,), dtype=np.int32),
            })
            for agent in self.possible_agents
        }
        
        # Action space
        # The agent can choose to move up, down, left, or right
        self.action_spaces = {
            agent: spaces.Discrete(4) 
            for agent in self.possible_agents
        }
        
        # List of visited spaces
        self._visited_spaces = {
            agent: []
            for agent in self.possible_agents
        }
        
        self._cumulative_rewards = {agent: 0.0 for agent in self.agents}
        self._agent_selector = agent_selector(self.agents)

        self.agent_colors = {
            "agent0": (255, 100, 100),
            "agent1": (100, 200, 255),
            "agent2": (100, 255, 120),
            "agent3": (255, 200, 100),
        }
        self.render_mode = render_mode
        self.seed()
        
    def observation_space(self, agent):
        """
        Returns a single agent's observation space. Part of Tianshou's formatting.
        """
        return self.observation_spaces[agent]

    def action_space(self, agent):
        """
        Returns a single agent's action space. Part of Tianshou's formatting.
        """
        return self.action_spaces[agent]
    
    def reset(self, seed=None, options=None):
        """
        Resets the environment.
        Rewards, visited spaces, and steps taken are cleared out. 
        Coin and agent positions are randomized.
        "termination" and "truncation" are set to False.
        
        :return: observations, debug information
        """
        self.seed(seed)
        self.current_step = 0
        self._visited_spaces = {
            agent: []
            for agent in self.possible_agents
        }
        
        # Randomly place agents in empty spaces
        self.agents = self.possible_agents[:]
        self.agent_positions = {
            agent: self._get_empty_cell([]) for agent in self.agents
        }
        for agent in self.agents:
            self._visited_spaces[agent].append(self.agent_positions[agent])
        
        # Randomly place coins in empty spaces
        occupied = list(self.agent_positions.values())
        self.coin_positions = {agent: [] for agent in self.agents}
        for agent in self.agents:
            for _ in range(self.num_coins):
                pos = self._get_empty_cell(occupied)
                self.coin_positions[agent].append(pos)
                occupied.append(pos)
        
        # Environment reset
        self._cumulative_rewards = {agent: 0.0 for agent in self.agents}
        self.rewards = {agent: 0.0 for agent in self.agents}
        self.terminations = {agent: False for agent in self.agents}
        self.truncations = {agent: False for agent in self.agents}
        self.infos = {agent: {} for agent in self.agents}
        
        self.global_terminated = False
        self.global_truncated = False
        
        self._agent_selector = agent_selector(self.agents)
        self.agent_selection = self._agent_selector.next()

        return self.observe(self.agent_selection), self.infos[self.agent_selection]
    
    def step(self, action):
        """
        One step in an episode. The current agent moves based on its chosen action.
        
        Reward structure:
        -0.01 per step
        -0.5 if the agent stays still (action attempts to move into a wall)
        -0.1 if the agent visits a space that they visited recently (such as repeatedly moving back and forth)
        +0.3 if the agent visits a new space (reward to drive exploration)
        
        +2.0 * [grid size] if the agent finds a coin (big reward that scales with grid size)
        
        +3.0 * [grid size] * [number of coins] if the agent collects all coins (success bonus)
        -1.0 * [grid size] * [number of coins] if max steps is reached (failure penalty)
        
        :param action: integer value within the agent's action space (0 to 3)
        
        :return: observations, rewards, episode termination (boolean), episode truncation (boolean), debug information
        """
        self._clear_rewards()
        self.current_step += 1
        agent = self.agent_selection
        
        # Movement
        x, y = self.agent_positions[agent]
        self.agent_positions[agent] = self._move(x, y, action)
        pos = self.agent_positions[agent]
        
        # Movement reward
        shared_reward = -0.01
        solo_reward = 0.0

        if pos == self._visited_spaces[agent][-1]:
            solo_reward -= 0.5
        elif pos in self._visited_spaces[agent][-4:]:
            solo_reward -= 0.1
        elif pos not in self._visited_spaces[agent]:
            solo_reward += 0.3
            self._visited_spaces[agent].append(pos)        
            
        # Coin collection reward
        if pos in self.coin_positions[agent]:
            solo_reward += 2.0*self.size
            self.coin_positions[agent].remove(pos)
        
        # Check for termination and truncation, plus success/failure reward
        all_coins_collected = all(len(v) == 0 for v in self.coin_positions.values())
        max_steps_reached = self.current_step >= self.max_steps
        
        if all_coins_collected:
            shared_reward += 3.0*self.size*self.num_coins*len(self.agents)  # Massive reward for total success.
            for a in self.agents:
                self.terminations[a] = True
            self.agents = []
        elif max_steps_reached:
            shared_reward -= 1.0*self.size*self.num_coins*len(self.agents) # Penalty for failing to find all coins
            for a in self.agents:
                self.truncations[a] = True
            self.agents = []
        
        # Assign rewards
        self.rewards[agent] += solo_reward
        for a in self.agents:
            self.rewards[a] += shared_reward
        
        self._accumulate_rewards()
        self.agent_selection = self._agent_selector.next()
        
        if all(self.terminations.values()):
            # Episode ends for ALL agents simultaneously
            return (self.observe(agent), 
                   self.rewards[agent], 
                   True, False, 
                   self.infos[agent])
        if all(self.truncations.values()):
            return (self.observe(agent), 
                   self.rewards[agent], 
                   False, True, 
                   self.infos[agent])
        
        if self.render_mode == "human":
            self._render_frame()
        self.infos = self._get_info()
        
        return (self.observe(self.agent_selection), 
        self.rewards[self.agent_selection], 
        self.terminations[self.agent_selection], 
        self.truncations[self.agent_selection], 
        self.infos[self.agent_selection])
    
    def joint_step(self, joint_actions):
        """
        Execute one synchronous step for all agents; used only by the centralized environment.
        
        Reward structure is nearly identical to the step() function, but now shared between both agents.
        
        :param joint_actions: A dictionary of actions per agent
        
        :return: rewards, episode termination (boolean), episode truncation (boolean)
        """
        self._clear_rewards()
        self.current_step += 1

        shared_reward = -0.01
        
        agent_actions = set([])
        for agent, action in joint_actions.items():
            agent_actions.add(action)
            x, y = self.agent_positions[agent]
            self.agent_positions[agent] = self._move(x, y, int(action))
            pos = self.agent_positions[agent]

            if pos == self._visited_spaces[agent][-1]:
                shared_reward -= 0.5
            if pos in self._visited_spaces[agent][-4:]:
                shared_reward -= 0.1
            if pos not in self._visited_spaces[agent]:
                shared_reward += 0.3
                self._visited_spaces[agent].append(pos)

            if pos in self.coin_positions[agent]:
                shared_reward += 2.0 * self.size
                self.coin_positions[agent].remove(pos)
        
        all_coins = all(len(v) == 0 for v in self.coin_positions.values())
        max_steps = self.current_step >= self.max_steps

        if all_coins:
            shared_reward += 3.0 * self.size * self.num_coins * len(self.agents)
            for agent in self.agents:
                self.terminations[agent] = True
        elif max_steps:
            shared_reward -= 1.0 * self.size * self.num_coins * len(self.agents)
            for agent in self.agents:
                self.truncations[agent] = True

        self._accumulate_rewards()
        terminated = all(self.terminations.values())
        truncated = all(self.truncations.values())
        return shared_reward, terminated, truncated
    
    def observe(self, agent):
        """
        Returns an agent's updated observation. Part of Tianshou's formatting.
        """
        return self._get_obs(agent)
    
    def _get_obs(self, agent):
        """
        Creates an agent's new observation after taking an action.
        
        The observation space has 2 parts: a grid of local observations and an
        array of "knowledge" that the agent is aware of.
        
        The local observations have 5 channels, which each handle a different
        aspect of the grid. The first handles walls, the second handles visited
        spaces, the third handles coins, and the fourth and fifth encode the 
        agent's global x and y coordinates.
        
        The knowledge observation is of the agent's global x and y coordinates,
        the number of individual coins the agent has left, and the total number
        of coins all agents have left.
        
        :return: full_obs, a dictionary of the local and knowledge observations.
        """
        full_obs = {}
        r = self._view_radius
        view_size = 2 * r + 1
        
        ax, ay = self.agent_positions[agent] # Agent global position
        
        # Local observation space
        view_box = np.zeros((5, view_size, view_size), dtype=np.float32)     
        for dx in range(-r, r+1): # Direction from agent
            for dy in range(-r, r+1): 
                x, y = ax+dx, ay+dy # Global position of viewbox cell
                lx, ly = dx+r, dy+r # Local position of viewbox cell
                
                # Channel 0: walls. If out of bounds, then 1. Otherwise, 0.
                if not (0 <= x < self.size and 0 <= y < self.size): 
                    view_box[0, lx, ly] = 1
                    continue
                
                # Channel 1: visitation.
                # If viewed cell is unvisited, then 1. If recently visited, then 0. Otherwise, 0.5.
                if (x, y) not in self._visited_spaces[agent]:
                    view_box[1, lx, ly] = 1
                elif (x, y) not in self._visited_spaces[agent][-4:]:
                    view_box[1, lx, ly] = 0.5

                # Channel 2: coins. Coins of the same color = 1, otherwise 0.
                if (x, y) in self.coin_positions[agent]:
                    view_box[2, lx, ly] = 1
        
        # Channels 3 and 4: global x and y
        view_box[3, :, :] = ax / self.size
        view_box[4, :, :] = ay / self.size
        full_obs["local"] = view_box
        
        # Innate knowledge: global x and y, individual number of coins left, total number of coins left.
        knowledge_box = np.zeros((4,), dtype=np.int32)
        knowledge_box[0] = self.agent_positions[agent][0]
        knowledge_box[1] = self.agent_positions[agent][1]
        knowledge_box[2] = len(self.coin_positions[agent])
        knowledge_box[3] = sum(len(v) for v in self.coin_positions.values())
        full_obs["knowledge"] = knowledge_box
        
        return full_obs
    
    def _move(self, x, y, action):
        """
        Converts an action and global coordinates to movement.
        Simply updates x and y unless the new position is out of bounds.
        
        :param x: agent's x coordinate
        :param y: agent's y coordinate
        :param action: agent's action
        
        :return: x, y after the agent's move.
        """
        if action == 0 and x > 0: #Left
            return x - 1, y
        if action == 1 and x < self.size - 1: #Right
            return x + 1, y
        if action == 2 and y > 0: #Up
            return x, y - 1
        if action == 3 and y < self.size - 1: #Down
            return x, y + 1
        return x, y
    
    def _clear_rewards(self):
        """
        Reset per-step rewards. Part of Tianshou's formatting.
        """
        for agent in self.agents:
            self.rewards[agent] = 0.0

    def _accumulate_rewards(self):
        """
        Accumulate rewards. Part of Tianshou's formatting.
        """
        for agent in self.agents:
            self._cumulative_rewards[agent] += self.rewards[agent]
    
    def seed(self, seed=None):
        """
        Sets the random numpy seed.
        """
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

    def _get_empty_cell(self, occupied):
        """
        Returns a random unoccupied cell. Used during reset to place agents and coins.
        
        :param occupied: list of currently occupied cells.
        
        :return: cell, a random unoccupied (x, y) coordinate in the grid.
        """
        while True:
            cell = (
                self.np_random.integers(0, self.size, dtype=int),
                self.np_random.integers(0, self.size, dtype=int)
            )
            if cell not in occupied:
                return cell
    
    def _get_info(self):
        """
        Returns some debugging information. Largely unused.
        """
        return {
            agent: {
                "location": self.agent_positions[agent],
                "coins_left": len(self.coin_positions.get(agent, [])),
            }
            for agent in self.agents
        }
    
    def render(self):
        """
        Clears all previous output and renders a new one as an image in a Jupyter Notebook.
        """
        frame = self._render_frame()
        clear_output(wait=True) 
        display(Image.fromarray(frame))
        return frame

    def _render_frame(self):
        """
        Draws the frame that is being rendered. Coins are color-coded by what agent should collect them.
        """
        if getattr(self, "window", None) is None:
            pygame.init()
            pygame.display.init()
            self.window_size = 500
            self.window = pygame.Surface((self.window_size, self.window_size))

        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))
        pix_square_size = self.window_size / self.size

        # Draw coins
        for agent in self.agents:
            color = self.agent_colors[agent]
            for (cx, cy) in self.coin_positions[agent]:
                pygame.draw.circle(
                    canvas, color,
                    ((cx + 0.5) * pix_square_size, (cy + 0.5) * pix_square_size),
                    pix_square_size / 3
                )

        # Draw agents
        for agent in self.agents:
            ax, ay = self.agent_positions[agent]
            color = self.agent_colors[agent]
            padding = pix_square_size * 0.15
            pygame.draw.rect(
                canvas, color,
                pygame.Rect(
                    (ax * pix_square_size + padding, ay * pix_square_size + padding),
                    (pix_square_size - 2 * padding, pix_square_size - 2 * padding)
                ),
            )

        # Draw grid
        for x in range(self.size + 1):
            pygame.draw.line(canvas, (0, 0, 0),
                             (0, pix_square_size * x),
                             (self.window_size, pix_square_size * x), width=1)
            pygame.draw.line(canvas, (0, 0, 0),
                             (pix_square_size * x, 0),
                             (pix_square_size * x, self.window_size), width=1)

        frame = pygame.surfarray.array3d(canvas)
        return np.transpose(frame, (1, 0, 2))
    
    def close(self):
        """
        Closes a PyGame display. Part of Tianshou's formatting.
        """
        if getattr(self, "window", None) is not None:
            pygame.display.quit()
            pygame.quit()
            self.window = None

## Centralized Environment Wrapper
A wrapper for the above grid world environment, where a single policy decides the actions for all agents and agents move at the same time.

In [5]:
class CentralizedEnv(gym.Env):
    """
    A centralized view of MultiAgentGridWorld, where one policy chooses the actions for all agents.
    """
    metadata = {"render_modes": ["human", "rgb_array"]}

    def __init__(self, size=8, num_agents=2, num_coins=2, view_radius=2, render_mode=None):
        """
        Initializes the environment in a manner similar to the MultiAgentGridWorld environment. 
        However, the observation spaces of all agents are stacked on top of each other.
        """
        super().__init__()
        self.env = MultiAgentGridWorld(size=size,
                                            num_agents=num_agents,
                                            num_coins=num_coins,
                                            view_radius=view_radius,
                                            render_mode=render_mode)
        self.view_size = 2 * view_radius + 1
        self.num_coins = num_coins
        self.channels_per_agent = 5  # your local tensor already has 5 channels
        self.obs_local_shape = (num_agents * self.channels_per_agent,
                                self.view_size,
                                self.view_size)
        self.obs_knowledge_dim = num_agents * 4  # each agent’s 4-element knowledge vector
        self.observation_space = spaces.Dict(
            {
                "local": spaces.Box(low=0.0, high=1.0,
                                    shape=self.obs_local_shape, dtype=np.float32),
                "knowledge": spaces.Box(low=0, high=self.env.size,
                                        shape=(self.obs_knowledge_dim,), dtype=np.float32),
            }
        )
        # joint action = cartesian product of each agent’s 4 moves → 4**num_agents discrete choices
        self.branch_sizes = [self.env.action_space(agent).n for agent in self.env.possible_agents]
        self.action_space = spaces.Discrete(int(np.prod(self.branch_sizes)))

    def _stack_obs(self):
        """
        Makes all agents observe the environment individually as in MultiAgentGridWorld, then stacks their observation spaces.
        For example, if an agent's normal local observation is a 5x5 matrix with 5 channels, the stacked local observation
        with 2 agents is a 5x5 matrix with 10 channels.
        
        :return: the stacked observation
        """
        local = np.zeros(self.obs_local_shape, dtype=np.float32)
        knowledge = np.zeros((self.obs_knowledge_dim,), dtype=np.float32)
        for idx, agent in enumerate(self.env.possible_agents):
            obs = self.env.observe(agent)
            c_start = idx * self.channels_per_agent
            c_end = c_start + self.channels_per_agent
            local[c_start:c_end] = obs["local"]
            k_start = idx * 4
            k_end = k_start + 4
            knowledge[k_start:k_end] = obs["knowledge"]
        return {"local": local, "knowledge": knowledge}

    def reset(self, seed=None, options=None):
        """
        Resets the environment by calling MultiAgentGridWorld's reset function.
        """
        obs, _ = self.env.reset(seed=seed, options=options)
        return self._stack_obs(), {}

    def step(self, joint_action):
        """
        Turns a numpy array of joint actions into a dictionary per agent.
        Then, calls MultiAgentGridWorld's joint_step function with that dictionary.
        
        :return: observations, rewards, episode termination, episode truncation, debugging information.
        """
        actions = np.unravel_index(joint_action, self.branch_sizes)
        action_dict = {
            agent: int(act)
            for agent, act in zip(self.env.possible_agents, actions)
        }
        reward, terminated, truncated = self.env.joint_step(action_dict)
        return self._stack_obs(), reward, terminated, truncated, {}

    def render(self):
        """
        Renders the environment by calling MultiAgentGridWorld's render function.
        """
        return self.env.render()

    def close(self):
        """
        Closes an environment display window by calling MultiAgentGridWorld's close function.
        """
        self.env.close()

### Neural Network
The neural network used to train the policy. It takes the observation space as input, and outputs an action (or set of actions, if using the centralized environment). The network is comprised of a CNN and Fully Connected Network to process separate parts of the observation, which both feed into one more Fully Connected Network.

In [6]:
class QNet(nn.Module):
    """
    Defines the neural network that controls the agent. 
    """
    def __init__(self, local_shape, knowledge_dim, joint_action_dim, coins, num_agents):
        """
        Creates the neural network. 
        The local observation space gets passed through a CNN to preserve spacial relations.
        The knowledge observation space gets passed through a Fully Connected Network.
        Both outputs are flattened and put through a final Fully Connected Network that assigns q-values to actions.
        
        :param local_shape: array containing the dimensions of the local observation space (C, H, W).
        :param knowledge_dim: the dimension of the knowledge observation space.
        :param action_dim: the dimension of the action space.
        :param coins: the number of coins per agent.
        :param num_agents: the number of agents.
        """
        super().__init__()
        c, h, w = local_shape
        self.num_coins = coins
        self.num_agents = num_agents
        self.local_conv = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        conv_out = 64 * h * w
        self.knowledge_mlp = nn.Sequential(
            nn.Linear(knowledge_dim, 256),
            nn.ReLU(),
        )
        fused = conv_out + 256
        self.q_head = nn.Sequential(
            nn.Linear(fused, 256),
            nn.ReLU(),
            nn.Linear(256, joint_action_dim),
        )

    
    def forward(self, obs, state=None, info={}):
        """
        Performs one pass through the entire network.
        
        :param obs: dictionary containing the local and knowledge observations of an agent.
        :param state: used for stateful policies, but not useful here.
        
        :return: a q-value array corresponding to the actions.
        """
        local = torch.as_tensor(obs["local"], dtype=torch.float32, device=self.q_head[-1].weight.device)
        knowledge = torch.as_tensor(obs["knowledge"], dtype=torch.float32, device=self.q_head[-1].weight.device)

        if local.ndim == 3:
            local = local.unsqueeze(0)  # Adds an extra "batch" dimension because PyTorch expects it [B,C,H,W]
            knowledge = knowledge.unsqueeze(0)

        # normalize coordinates to [0, 1]
        knowledge_norm = knowledge.clone()
        knowledge[:, 0:2] /= obs["local"].shape[-1]          # agent coordinates
        knowledge[:, 2] /= self.num_coins                    # solo coins
        knowledge[:, 3] /= self.num_coins * self.num_agents  # total coins

        local_feat = self.local_conv(local)
        knowledge_feat = self.knowledge_mlp(knowledge)
        fused = torch.cat([local_feat, knowledge_feat], dim=-1)

        q = self.q_head(fused)
        return q, state